# 3회차 실습: Genesis + Franka Starter for Manipulation

이 노트북은 학생 배포용 실습 자료입니다.

이번 시간부터는 실제 simulator 안에서 Franka를 움직입니다.
이번 실습의 목표는 `주어진 3개의 target pose를 부드럽게 순서대로 방문하는 motion sequence`를 완성하는 것입니다.

이번 노트북에서 제공되는 것:
- Genesis scene 생성 코드
- Franka / table / cube / obstacle / pose marker 생성 코드
- 현재 state 읽기, gripper 제어, joint trajectory 실행 함수

이번 노트북에서 직접 구현할 것:
- `solve_pose_ik()`
- `interpolate_joint_trajectory()`
- `plan_pose_sequence()`

실습 방법:
1. 위에서부터 셀을 순서대로 실행합니다.
2. starter code는 읽고 이해하되, `TODO(student)`가 있는 함수만 직접 구현합니다.
3. 마지막 과제 셀에서 3개 pose를 순서대로 방문하는지 확인합니다.


## 오늘 할 일

이번 시간의 과제는 단순합니다.

1. scene 안에 있는 Franka와 target pose marker를 확인하기
2. target pose를 built-in IK로 joint target으로 바꾸기
3. joint trajectory를 smooth하게 만들어 pose들을 순서대로 방문하기
4. 각 pose에서 1초 동안 정지시키기

이번 시간에는 pick-and-place 전체보다, `pose sequence를 안정적으로 실행하는 기본 틀`을 만드는 것이 더 중요합니다.


## 0. Imports and Utility Functions

In [ ]:
import numpy as np
import genesis as gs
from dataclasses import dataclass, field

np.set_printoptions(precision=4, suppress=True)


def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        return x.numpy().astype(float)
    return np.asarray(x, dtype=float)


## 1. Starter Scene 만들기

In [ ]:
@dataclass
class Week3Config:
    dt: float = 0.01
    settle_steps: int = 120
    hold_time_sec: float = 1.0
    steps_per_segment: int = 120
    gripper_open: float = 0.04
    gripper_closed: float = 0.0
    home_qpos: tuple[float, ...] = (
        0.0,
        -0.65,
        0.0,
        -2.0,
        0.0,
        1.35,
        0.78,
        0.04,
        0.04,
    )
    table_pos: tuple[float, float, float] = (0.45, 0.0, 0.20)
    table_size: tuple[float, float, float] = (0.70, 1.10, 0.40)
    cube_pos: tuple[float, float, float] = (0.58, -0.18, 0.43)
    cube_size: float = 0.045
    obstacle_pos: tuple[float, float, float] = (0.52, 0.0, 0.50)
    obstacle_size: tuple[float, float, float] = (0.06, 0.26, 0.12)
    pose_markers: list[tuple[tuple[float, float, float], tuple[float, float, float]]] = field(
        default_factory=lambda: [
            ((0.52, -0.22, 0.58), (0.95, 0.30, 0.30)),
            ((0.62,  0.00, 0.48), (0.30, 0.80, 0.30)),
            ((0.44,  0.22, 0.60), (0.30, 0.40, 0.95)),
        ]
    )


cfg = Week3Config()


In [ ]:
def setup_franka(robot):
    robot.set_dofs_kp(np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]))
    robot.set_dofs_kv(np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]))
    robot.set_dofs_force_range(
        np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
        np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
    )


def build_scene(cfg, show_viewer=True, backend=gs.gpu):
    gs.init(backend=backend)

    scene = gs.Scene(
        viewer_options=gs.options.ViewerOptions(
            camera_pos=(2.4, -1.2, 1.5),
            camera_lookat=(0.45, 0.0, 0.45),
            camera_fov=35,
            max_FPS=60,
        ),
        sim_options=gs.options.SimOptions(dt=cfg.dt),
        rigid_options=gs.options.RigidOptions(
            box_box_detection=True,
            enable_collision=True,
            enable_joint_limit=True,
        ),
        show_viewer=show_viewer,
    )

    scene.add_entity(gs.morphs.Plane(), name="ground")
    scene.add_entity(gs.morphs.Box(pos=cfg.table_pos, size=cfg.table_size, fixed=True), surface=gs.surfaces.Plastic(color=(0.76, 0.73, 0.68)), name="table")
    scene.add_entity(gs.morphs.Box(pos=cfg.cube_pos, size=(cfg.cube_size, cfg.cube_size, cfg.cube_size)), material=gs.materials.Rigid(rho=250), surface=gs.surfaces.Plastic(color=(0.85, 0.28, 0.28)), name="cube")
    scene.add_entity(gs.morphs.Box(pos=cfg.obstacle_pos, size=cfg.obstacle_size, fixed=True), surface=gs.surfaces.Plastic(color=(0.20, 0.45, 0.92)), name="obstacle")

    marker_size = (0.035, 0.035, 0.035)
    for idx, (pos, color) in enumerate(cfg.pose_markers):
        scene.add_entity(
            gs.morphs.Box(pos=pos, size=marker_size, fixed=True, collision=False),
            surface=gs.surfaces.Plastic(color=color),
            name=f"pose_marker_{idx}",
        )

    robot = scene.add_entity(gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"), name="franka")

    scene.build()
    setup_franka(robot)
    robot.set_qpos(np.array(cfg.home_qpos))
    for _ in range(cfg.settle_steps):
        scene.step()

    return scene, robot


In [ ]:
scene, robot = build_scene(cfg, show_viewer=True)
ee_link = robot.get_link("hand")
print("Scene ready.")


## 2. 현재 robot state 읽기

In [ ]:
def read_robot_state(robot, ee_link):
    q = to_numpy(robot.get_qpos())
    ee_pos = to_numpy(ee_link.get_pos())
    ee_quat = to_numpy(ee_link.get_quat())
    return q, ee_pos, ee_quat


q_now, ee_pos_now, ee_quat_now = read_robot_state(robot, ee_link)
print("Current q =", q_now)
print("Current ee position =", ee_pos_now)
print("Current ee quaternion =", ee_quat_now)


## 3. Starter Utility: joint command와 gripper command

In [ ]:
def hold_current_target(scene, robot, q_target, steps):
    for _ in range(steps):
        robot.control_dofs_position(q_target)
        scene.step()


def set_gripper_open(q, cfg):
    q = np.array(q, dtype=float, copy=True)
    q[-2:] = cfg.gripper_open
    return q


def set_gripper_closed(q, cfg):
    q = np.array(q, dtype=float, copy=True)
    q[-2:] = cfg.gripper_closed
    return q


def open_gripper(scene, robot, cfg, hold_steps=80):
    q = set_gripper_open(to_numpy(robot.get_qpos()), cfg)
    hold_current_target(scene, robot, q, hold_steps)
    return q


def close_gripper(scene, robot, cfg, hold_steps=80):
    q = set_gripper_closed(to_numpy(robot.get_qpos()), cfg)
    hold_current_target(scene, robot, q, hold_steps)
    return q


def execute_joint_trajectory(scene, robot, q_traj, hold_steps=0):
    q_last = None
    for q in q_traj:
        q_last = np.asarray(q, dtype=float)
        robot.control_dofs_position(q_last)
        scene.step()
    if q_last is not None and hold_steps > 0:
        hold_current_target(scene, robot, q_last, hold_steps)
    return q_last


## 4. 구현 포인트 1: target pose를 joint target으로 바꾸기

이번 실습에서는 IK를 손으로 유도하지 않습니다.
대신 Genesis 내장 IK를 호출해서 `target pose -> q_goal` 변환만 구현하면 됩니다.

이 함수에서 해야 할 일은 아래 세 줄 정도입니다.

1. `robot.inverse_kinematics(...)` 호출
2. 결과를 numpy array로 바꾸기
3. gripper width를 마지막 두 joint에 반영하기


In [ ]:
def solve_pose_ik(robot, ee_link, target_pos, target_quat, gripper_width=0.04):
    # TODO(student): Genesis built-in IK를 사용해서 q_goal을 구해 보세요.
    # 1. robot.inverse_kinematics(...) 호출
    # 2. to_numpy(...)로 변환
    # 3. q_goal[-2:] = gripper_width
    raise NotImplementedError('Implement solve_pose_ik() before running the next cells.')


## 5. 구현 포인트 2: smooth joint trajectory 만들기

이번 과제의 smoothness는 아주 복잡하게 다룰 필요 없습니다.
joint-space에서 시작점과 끝점을 cubic time scaling으로 보간하면 충분합니다.

사용할 식은 아래와 같습니다.

$$
s(	au) = 3	au^2 - 2	au^3, \quad 	au \in [0,1]
$$

$$
q(	au) = (1-s) q_{start} + s q_{goal}
$$

즉, `tau`, `s`, `q_traj` 세 줄을 잘 만들면 됩니다.


In [ ]:
def interpolate_joint_trajectory(q_start, q_goal, num_steps):
    # TODO(student): cubic time scaling으로 smooth joint trajectory를 만들어 보세요.
    # 1. tau = np.linspace(0, 1, num_steps)
    # 2. s = 3 * tau**2 - 2 * tau**3
    # 3. q_traj = (1-s) q_start + s q_goal
    raise NotImplementedError('Implement interpolate_joint_trajectory() before running the next cells.')


## 6. 구현 포인트 3: pose sequence planner 만들기

이번 과제의 핵심 함수입니다.
세 개의 pose를 순서대로 처리하면서, 각 구간마다 아래 순서를 반복하면 됩니다.

1. 현재 q에서 시작한다.
2. pose 하나를 IK로 `q_goal`로 바꾼다.
3. `q_current -> q_goal` trajectory를 만든다.
4. trajectory를 실행한다.
5. 마지막 pose에서 1초 정지한다.
6. 다음 구간을 위해 `q_current = q_goal`로 갱신한다.


In [ ]:
def plan_pose_sequence(scene, robot, ee_link, pose_list, cfg, gripper_width=None):
    # TODO(student): 여러 pose를 순서대로 방문하는 planner를 완성해 보세요.
    # 힌트:
    # - q_current = to_numpy(robot.get_qpos())
    # - hold_steps = int(cfg.hold_time_sec / cfg.dt)
    # - 각 pose마다 solve_pose_ik -> interpolate_joint_trajectory -> execute_joint_trajectory 순서로 진행
    raise NotImplementedError('Implement plan_pose_sequence() before running the next cells.')


## 7. 4회차 실습 과제: 3개 pose를 순서대로 방문하기

In [ ]:
downward_quat = np.array([0.0, 1.0, 0.0, 0.0])
pose_sequence = [
    {"name": "pose_0", "pos": np.array(cfg.pose_markers[0][0], dtype=float), "quat": downward_quat},
    {"name": "pose_1", "pos": np.array(cfg.pose_markers[1][0], dtype=float), "quat": downward_quat},
    {"name": "pose_2", "pos": np.array(cfg.pose_markers[2][0], dtype=float), "quat": downward_quat},
]

open_gripper(scene, robot, cfg, hold_steps=60)
q_goals = plan_pose_sequence(scene, robot, ee_link, pose_sequence, cfg, gripper_width=cfg.gripper_open)
print("Finished all target poses.")


## 제출 전 체크

아래 항목을 스스로 확인해 보세요.

- `solve_pose_ik()`가 target pose를 joint target으로 잘 바꾸는가?
- `interpolate_joint_trajectory()`가 너무 급하지 않은 smooth motion을 만드는가?
- `plan_pose_sequence()`가 3개 pose를 순서대로 방문하는가?
- 각 pose에서 1초 동안 정지하는가?
